# XGBoost FPR predictor

Train a single XGBoost regressor that predicts the average reservoir pressure (FPR)
per timestep from the SPE9 + OPM Flow dataset. The notebook runs top-to-bottom on
Google Colab and locally without modification.

Pipeline:

1. Load `dataset.csv` (one row per (sim, timestep), 17 columns).
2. Drop columns that leak the target or have zero variance.
3. Split by simulation (not by row) so the test set holds out whole sims.
4. Train an XGBoost regressor with early stopping.
5. Report MAE / RMSE / R^2 on the held-out sims and run a 5-fold GroupKFold
   robustness check.
6. Plot diagnostics: predicted vs actual, residuals, per-sim traces, feature
   importance, learning curves.
7. Run a small manual hyperparameter sweep.
8. Save the trained model.


## 1. Setup


In [ ]:
# Pin xgboost >= 2.0 for early_stopping_rounds in the constructor.
# Safe to re-run; pip is silent on a no-op.
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "xgboost>=2.0", "scikit-learn", "pandas", "matplotlib"], check=True)


In [ ]:
import io
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=UserWarning)
RNG_SEED = 42


## 2. Load `dataset.csv`

The notebook tries the local path first and falls back to a Colab upload prompt or Drive mount.


In [ ]:
def load_dataset() -> pd.DataFrame:
    candidates = [
        Path("datasets/dataset_spe9.csv"),
        Path("../datasets/dataset_spe9.csv"),
        Path("dataset.csv"),
        Path("../dataset.csv"),
        Path("/content/dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            print(f"loaded from {path}")
            return pd.read_csv(path)

    try:
        from google.colab import files  # type: ignore
    except ImportError:
        raise FileNotFoundError(
            "datasets/dataset_spe9.csv not found in working directory or parent. "
            "Place it next to the notebook or run on Colab to upload."
        )
    print("dataset not found locally; opening Colab upload widget...")
    uploaded = files.upload()
    return pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))


df = load_dataset()
print("shape:", df.shape)
df.head()

## 3. Quick EDA


In [ ]:
print("columns:", list(df.columns))
print()
print(df.describe().T[["mean", "std", "min", "max"]].round(3))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df["Presion_Reservorio_psi"], bins=40, color="steelblue", edgecolor="black")
ax[0].set_title("FPR distribution (all rows)")
ax[0].set_xlabel("FPR (psi)")
ax[0].set_ylabel("count")
ax[0].grid(alpha=0.3)

per_sim_min = df.groupby("sim_id")["Presion_Reservorio_psi"].min()
per_sim_max = df.groupby("sim_id")["Presion_Reservorio_psi"].max()
ax[1].scatter(per_sim_max, per_sim_min, s=22, alpha=0.7)
ax[1].set_xlabel("FPR_max per sim")
ax[1].set_ylabel("FPR_min per sim")
ax[1].set_title("Per-simulation pressure envelope")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Leakage and zero-variance audit

Three columns are computed from FPR by deterministic interpolation in
`scripts/pvt_tables.py`: `Bo_rb_stb`, `Bg_rb_scf`, `Rs_scf_stb`. Including
them as features lets the model invert the PVT lookup and trivially recover
FPR. They have to be dropped.

Two more columns (`Espesor_Neto_m`, `Area`) are constants in SPE9, so they
carry no information and are dropped too.


In [ ]:
LEAKAGE_COLS = ["Bo_rb_stb", "Bg_rb_scf", "Rs_scf_stb"]
ZERO_VAR_COLS = ["Espesor_Neto_m", "Area"]
EXCLUDED = LEAKAGE_COLS + ZERO_VAR_COLS

audit_rows = []
for col in EXCLUDED:
    audit_rows.append(
        {
            "column": col,
            "variance": df[col].var(),
            "reason": "PVT-derived from FPR (leakage)" if col in LEAKAGE_COLS else "constant in SPE9 (zero variance)",
        }
    )
pd.DataFrame(audit_rows).set_index("column")


## 5. Feature and target selection


In [ ]:
FEATURES = [
    "Porosidad",
    "Permeabilidad_mD",
    "Presion_Burbuja_psi",
    "Caudal_Prod_Petroleo_bbl",
    "Caudal_Prod_Gas_Mpc",
    "Caudal_Iny_Agua_bbl",
    "Prod_Acumulada_Petroleo",
    "Prod_Acumulada_Gas",
    "Prod_Acumulada_Agua",
    "Iny_Acumulada_Agua",
]
TARGET = "Presion_Reservorio_psi"

X_all = df[FEATURES].to_numpy()
y_all = df[TARGET].to_numpy()
groups_all = df["sim_id"].to_numpy()
print("X shape:", X_all.shape, " | y shape:", y_all.shape)


## 6. Sim-level split (80 train / 20 test)

Random row-level splitting leaks because rows from the same simulation are
strongly correlated in time. We hold out whole simulations.


In [ ]:
rng = np.random.default_rng(RNG_SEED)
sim_ids = df["sim_id"].unique().copy()
rng.shuffle(sim_ids)

n_test_sims = 20
test_sims = sim_ids[:n_test_sims]
train_sims_full = sim_ids[n_test_sims:]

train_mask = df["sim_id"].isin(train_sims_full).to_numpy()
test_mask = df["sim_id"].isin(test_sims).to_numpy()

X_train_full = X_all[train_mask]
y_train_full = y_all[train_mask]
groups_train_full = groups_all[train_mask]
X_test = X_all[test_mask]
y_test = y_all[test_mask]

# Carve out 16 sims from training for early-stopping validation
val_sims = train_sims_full[:16]
val_mask_in_full = np.isin(groups_train_full, val_sims)
X_val = X_train_full[val_mask_in_full]
y_val = y_train_full[val_mask_in_full]
X_train = X_train_full[~val_mask_in_full]
y_train = y_train_full[~val_mask_in_full]

print(f"train sims: {len(train_sims_full) - len(val_sims)}, val sims: {len(val_sims)}, test sims: {len(test_sims)}")
print(f"train rows: {len(X_train)}, val rows: {len(X_val)}, test rows: {len(X_test)}")


## 7. Train baseline XGBoost


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RNG_SEED,
    tree_method="hist",
    early_stopping_rounds=30,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)
print("best iteration:", model.best_iteration)
print("trees grown:", model.n_estimators)


## 8. Evaluate on held-out sims


In [ ]:
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
r2 = r2_score(y_test, preds)

baseline_pred = np.full_like(y_test, np.median(y_train), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

results = pd.DataFrame(
    {
        "metric": ["MAE", "RMSE", "R^2", "MAE / mean(y_test)", "Baseline MAE (median)"],
        "value": [f"{mae:.2f} psi", f"{rmse:.2f} psi", f"{r2:.4f}",
                  f"{mae / y_test.mean():.3%}", f"{baseline_mae:.2f} psi"],
    }
).set_index("metric")
results


## 9. Diagnostics

### 9.1 Predicted vs actual


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
test_groups = groups_all[test_mask]
sc = ax.scatter(y_test, preds, c=test_groups, cmap="tab20", s=10, alpha=0.7)
lo = float(min(y_test.min(), preds.min()))
hi = float(max(y_test.max(), preds.max()))
ax.plot([lo, hi], [lo, hi], "k--", lw=1.0, label="y = x")
ax.set_xlabel("FPR true (psi)")
ax.set_ylabel("FPR predicted (psi)")
ax.set_title(f"Predicted vs actual on {n_test_sims} held-out sims")
ax.grid(alpha=0.3)
ax.legend()
plt.colorbar(sc, ax=ax, label="sim_id")
plt.tight_layout()
plt.show()


### 9.2 Residuals


In [ ]:
residuals = y_test - preds
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(residuals, bins=40, color="slategray", edgecolor="black")
axes[0].axvline(0, color="firebrick", lw=1.0)
axes[0].set_title(f"Residuals (mean={residuals.mean():+.1f}, std={residuals.std():.1f} psi)")
axes[0].set_xlabel("y_true - y_pred (psi)")
axes[0].grid(alpha=0.3)

axes[1].scatter(preds, residuals, s=8, alpha=0.4)
axes[1].axhline(0, color="firebrick", lw=1.0)
axes[1].set_xlabel("Predicted FPR (psi)")
axes[1].set_ylabel("Residual (psi)")
axes[1].set_title("Residuals vs predicted FPR")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 9.3 Per-sim residual traces


In [ ]:
test_df = df.loc[test_mask].copy()
test_df["predicted"] = preds
test_df["step"] = test_df.groupby("sim_id").cumcount()

per_sim_min = test_df.groupby("sim_id")["Presion_Reservorio_psi"].min().sort_values()
chosen = [
    per_sim_min.index[0],
    per_sim_min.index[len(per_sim_min) // 3],
    per_sim_min.index[2 * len(per_sim_min) // 3],
    per_sim_min.index[-1],
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, sid in zip(axes.flat, chosen):
    sub = test_df[test_df["sim_id"] == sid].sort_values("step")
    ax.plot(sub["step"], sub["Presion_Reservorio_psi"], color="firebrick", label="true", lw=1.5)
    ax.plot(sub["step"], sub["predicted"], color="steelblue", linestyle="--", label="pred", lw=1.2)
    ax.set_title(f"sim_{sid:04d}")
    ax.set_xlabel("Report step")
    ax.set_ylabel("FPR (psi)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle("True vs predicted FPR over time, four held-out simulations")
plt.tight_layout()
plt.show()


### 9.4 Feature importance


In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
importance.plot.barh(ax=ax, color="darkslateblue")
ax.set_xlabel("Gain importance")
ax.set_title("Feature importance (gain)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()
print(importance.sort_values(ascending=False).round(4).to_string())


### 9.5 Learning curves


In [ ]:
evals = model.evals_result()
train_rmse = evals["validation_0"]["rmse"]
val_rmse = evals["validation_1"]["rmse"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(train_rmse, label="train RMSE", color="steelblue")
ax.plot(val_rmse, label="val RMSE", color="firebrick")
ax.axvline(model.best_iteration, color="k", linestyle=":", lw=1.0, label=f"best iter = {model.best_iteration}")
ax.set_xlabel("Boosting round")
ax.set_ylabel("RMSE (psi)")
ax.set_title("Learning curves")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 10. Robustness check: 5-fold GroupKFold


In [ ]:
gkf = GroupKFold(n_splits=5)
fold_results = []
for fold_idx, (tr, te) in enumerate(gkf.split(X_all, y_all, groups=groups_all)):
    m = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RNG_SEED,
        tree_method="hist",
    )
    m.fit(X_all[tr], y_all[tr], verbose=False)
    p = m.predict(X_all[te])
    fold_results.append(
        {
            "fold": fold_idx,
            "MAE": mean_absolute_error(y_all[te], p),
            "RMSE": float(np.sqrt(mean_squared_error(y_all[te], p))),
            "R2": r2_score(y_all[te], p),
        }
    )
fold_df = pd.DataFrame(fold_results)
print(fold_df.round(3).to_string(index=False))
print()
print(f"MAE  mean = {fold_df['MAE'].mean():.2f} psi, std = {fold_df['MAE'].std():.2f} psi")
print(f"R^2  mean = {fold_df['R2'].mean():.4f}, std = {fold_df['R2'].std():.4f}")


## 11. Hyperparameter sweep

A small 3 x 3 grid over `max_depth` and `learning_rate`. Trains a fresh model
per combination on the same train/val/test split as section 6 and reports the
held-out test MAE.


In [ ]:
sweep_rows = []
for max_depth in [4, 6, 8]:
    for lr in [0.03, 0.05, 0.1]:
        m = xgb.XGBRegressor(
            n_estimators=600,
            learning_rate=lr,
            max_depth=max_depth,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RNG_SEED,
            tree_method="hist",
            early_stopping_rounds=30,
        )
        m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        p = m.predict(X_test)
        sweep_rows.append(
            {
                "max_depth": max_depth,
                "learning_rate": lr,
                "best_iter": m.best_iteration,
                "test_MAE": mean_absolute_error(y_test, p),
                "test_R2": r2_score(y_test, p),
            }
        )
sweep_df = pd.DataFrame(sweep_rows).sort_values("test_MAE").reset_index(drop=True)
sweep_df.round(4)


## 12. Save the trained baseline model


In [ ]:
model.save_model("xgboost_fpr.json")
print("saved xgboost_fpr.json")

try:
    from google.colab import files  # type: ignore
    files.download("xgboost_fpr.json")
except ImportError:
    pass


## 13. Notes

* **Leakage filter applied**: `Bo_rb_stb`, `Bg_rb_scf`, `Rs_scf_stb` are dropped
  because they are computed from FPR. Without that filter the test R^2 is
  effectively 1 and the model has not learned any reservoir physics.
* **Sim-level split applied**: random row splits give wildly optimistic test
  metrics on this dataset because of intra-simulation autocorrelation. Holding
  out whole sims is the only honest evaluation.
* **`p_init` marginal is non-uniform** (rejection sampling enforces
  `p_init >= Pb_new + 200`). Models that condition on `p_init` should expect
  fewer training points at low pressures.
* The cumulative columns (especially `Prod_Acumulada_Gas`) are the dominant
  predictors; they encode depletion state in a way that lets a tree-based,
  non-sequential model recover Pr without needing lag features.
* If results regress sharply, suspect (1) a row-shuffle split, (2) PVT columns
  re-introduced by mistake, (3) the LHS sample being too narrow on some lever.
